In [0]:
bronze_path = f"/Volumes/lakehouse_portfolio/bronze/breweries_raw/year=2026/month=03/day=27/"

#breweries_df = spark.read.option.("multiline",  "true")json(bronze_path)
breweries_df = spark.read.option("multiLine", "true").json(bronze_path)


print(f"Total de Registros lidos: {breweries_df.count()}")
breweries_df.printSchema()
breweries_df.show(5, truncate=False)



Total de Registros lidos: 9459
root
 |-- address_1: string (nullable = true)
 |-- address_2: string (nullable = true)
 |-- address_3: string (nullable = true)
 |-- brewery_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- name: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- state: string (nullable = true)
 |-- state_province: string (nullable = true)
 |-- street: string (nullable = true)
 |-- website_url: string (nullable = true)

+---------------------------+---------+---------+------------+--------------+-------------+------------------------------------+---------------+----------------+-----------------------+----------+-----------+---------+--------------+---------------------------+----------------------------+
|address_1                  |addres

In [0]:
from pyspark.sql import functions as F
breweries_df.select([
                     F.count(F.when(F.col(c).isNull(), c)).alias(c)
                     for c in breweries_df.columns
                    ]).show()

+---------+---------+---------+------------+----+-------+---+--------+---------+----+-----+-----------+-----+--------------+------+-----------+
|address_1|address_2|address_3|brewery_type|city|country| id|latitude|longitude|name|phone|postal_code|state|state_province|street|website_url|
+---------+---------+---------+------------+----+-------+---+--------+---------+----+-----+-----------+-----+--------------+------+-----------+
|      739|     9182|     9420|           0|   0|      0|  0|    2422|     2422|   0| 1018|          0|    0|             0|   739|       1227|
+---------+---------+---------+------------+----+-------+---+--------+---------+----+-----+-----------+-----+--------------+------+-----------+



In [0]:
silver_df = (
    breweries_df
    .drop("address_2", "address_3", "street")
    .filter(F.col("name").isNotNull())
    .filter(F.col("state").isNotNull())
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("state", F.trim(F.col("state")))
    .withColumn("brewery_type", F.lower(F.col("brewery_type")))
)
print(f"Bronze: {breweries_df.count()} registros")
print(f"Silver: {silver_df.count()} registros")
silver_df.show(5, truncate=False)

Bronze: 9459 registros
Silver: 9459 registros
+---------------------------+------------+--------------+-------------+------------------------------------+---------------+----------------+-----------------------+----------+-----------+---------+--------------+----------------------------+
|address_1                  |brewery_type|city          |country      |id                                  |latitude       |longitude       |name                   |phone     |postal_code|state    |state_province|website_url                 |
+---------------------------+------------+--------------+-------------+------------------------------------+---------------+----------------+-----------------------+----------+-----------+---------+--------------+----------------------------+
|1716 Topeka St             |micro       |Norman        |United States|5128df48-79fc-4f0f-8b52-d06be54d0cec|35.25738891    |-97.46818222    |(405) Brewing Co       |4058160490|73069-8224 |Oklahoma |Oklahoma      |http://www.4

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("state") \
    .saveAsTable("lakehouse_portfolio.silver.breweries")
print("Tabela Silver salva com sucesso!")
print("Local: lakehouse_portfolio.silver.breweries")

Tabela Silver salva com sucesso!
Local: lakehouse_portfolio.silver.breweries


In [0]:
# Lê a tabela que acabou de salvar e valida
tabela_silver = spark.table("lakehouse_portfolio.silver.breweries")

print(f"✅ Total de registros na Silver: {tabela_silver.count()}")
print(f"📊 Estados distintos: {tabela_silver.select('state').distinct().count()}")
print(f"🍺 Tipos de cervejaria: {tabela_silver.select('brewery_type').distinct().count()}")

# Distribuição por tipo
tabela_silver.groupBy("brewery_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

✅ Total de registros na Silver: 9459
📊 Estados distintos: 150
🍺 Tipos de cervejaria: 14
+------------+-----+
|brewery_type|count|
+------------+-----+
|       micro| 5076|
|     brewpub| 2649|
|    planning|  649|
|      closed|  362|
|    regional|  239|
|    contract|  184|
|       large|  117|
|  proprietor|   68|
|     taproom|   45|
|         bar|   37|
|        nano|   22|
|      cidery|    7|
|  beergarden|    3|
|    location|    1|
+------------+-----+

